In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/otto-recommender-system/sample_submission.csv
/kaggle/input/competitions/otto-recommender-system/test.jsonl
/kaggle/input/competitions/otto-recommender-system/train.jsonl


In [2]:
import json
import os
import polars as pl
import numpy as np

# 路径映射
DATA_DIR = '/kaggle/input/competitions/otto-recommender-system'
OUT_DIR = '/kaggle/working'

# 字符串到 uint8 的映射 (这是为你节省上 GB 内存的关键)
type_labels = {'clicks': 0, 'carts': 1, 'orders': 2}

数据降维 高效利用云空间

In [3]:
def process_data_in_chunks(file_path, chunk_size=100000):
    """流式读取 JSONL，拍平字典，并按 chunk_size 吐出数据"""
    with open(file_path, 'r') as f:
        chunk = []
        for line in f:
            # 解析单行 JSON
            data = json.loads(line)
            session_id = data['session']
            
            # 遍历事件列表，将嵌套结构拍平
            for event in data['events']:
                chunk.append({
                    'session': session_id,
                    'aid': event['aid'],
                    'ts': event['ts'],
                    'type': type_labels[event['type']] # 此时直接转为 0, 1, 2
                })
            
            # 达到块大小时，释放一波数据
            if len(chunk) >= chunk_size:
                yield chunk
                chunk = []
                
        # 吐出最后剩下的尾巴数据
        if chunk:
            yield chunk

数据块生成 整理jsonl中的数据

In [4]:
def convert_to_parquet(input_file, output_name):
    output_path = os.path.join(OUT_DIR, output_name)
    
    print(f"开始处理 {input_file}...")
    for i, chunk_data in enumerate(process_data_in_chunks(input_file)):
        
        # 将普通的 Python List[Dict] 转为极速的 Polars DataFrame
        df = pl.DataFrame(chunk_data)
        
        # 实施残酷的降维打击！
        df = df.with_columns([
            pl.col('session').cast(pl.Int32), 
            pl.col('aid').cast(pl.Int32),
            # 时间戳长度判断：如果是毫秒级时间戳(13位)，Int32装不下，必须用Int64。
            # 但如果是秒级(10位)，可以用 Int32。OTTO数据集是毫秒，所以此处保留Int64。
            pl.col('ts').cast(pl.Int64),
            pl.col('type').cast(pl.Int8)
        ])
        
        # 追加写入 Parquet 文件 (利用 pyarrow 引擎)
        import pyarrow as pa
        import pyarrow.parquet as pq
        
        table = df.to_arrow()
        if i == 0:
            # 第一个 chunk，创建文件并记录 schema
            writer = pq.ParquetWriter(output_path, table.schema)
            writer.write_table(table)
        else:
            # 随后的 chunk，向文件追加写入
            writer.write_table(table)
            
        if i % 100 == 0:
            print(f"已处理 {i} 个数据块 (约 {i * 100000} 行交互记录)")
            
    writer.close()
    print(f"处理完成！文件已保存至 {output_path}")

# 执行转换
convert_to_parquet(os.path.join(DATA_DIR, 'test.jsonl'), 'test.parquet')

开始处理 /kaggle/input/competitions/otto-recommender-system/test.jsonl...
已处理 0 个数据块 (约 0 行交互记录)
处理完成！文件已保存至 /kaggle/working/test.parquet
